*0.2 Math / ML basics*

# softmax

**The situation.** A router decides which team a ticket goes to. A classifier gives raw scores: billing 2.1, technical 0.4, sales −1.3. The product manager asks: "how confident is it?" A score of 2.1 is not an answer. You need probabilities — numbers between 0 and 1 that add up to 1.

**Softmax.** Take *e* to the power of each score (this makes every number positive and stretches the gaps), then divide by the total so they sum to 1. The biggest score gets the biggest share. Every language model does exactly this to its raw scores before it picks the next word.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import torch

teams = ["billing", "technical", "sales"]
scores = torch.tensor([2.1, 0.4, -1.3])  # raw scores from a classifier ("logits")

probabilities = torch.softmax(scores, dim=0)
for team, probability in zip(teams, probabilities):
    print(f"{team:<10} {probability:.1%}")
print("sum:", round(float(probabilities.sum()), 6))
assert abs(float(probabilities.sum()) - 1.0) < 1e-6

billing    82.2%
technical  15.0%
sales      2.7%
sum: 1.0


**Reading the output.** Billing about 82%, technical about 15%, sales about 3%. Same order as the raw scores, but now an answer the product manager can use — and a threshold you can set ("route automatically above 80%, otherwise ask a person").

**Scale changes confidence.** Double every score and the gaps double, so the winner takes more. Halve them and the split gets flatter. This is what *temperature* does to a language model, four items from now.

In [3]:
for scale in (0.5, 1.0, 2.0):
    shares = []
    for p in torch.softmax(scores * scale, dim=0).tolist():
        shares.append(f"{p:.1%}")
    print(f"scores × {scale}: ", shares)
sharper = torch.softmax(scores * 2.0, dim=0)[0]
flatter = torch.softmax(scores * 0.5, dim=0)[0]
assert sharper > probabilities[0] > flatter

scores × 0.5:  ['62.1%', '26.5%', '11.3%']
scores × 1.0:  ['82.2%', '15.0%', '2.7%']
scores × 2.0:  ['96.7%', '3.2%', '0.1%']


```
scores      [ 2.1,  0.4, -1.3 ]
e^score     [ 8.2,  1.5,  0.3 ]      all positive, gaps stretched
÷ total     [ 0.82, 0.15, 0.03 ]     add up to 1  → probabilities
```

**The rule to remember.** Softmax turns any list of scores into probabilities. It is the last step of every classifier and of every next-word prediction.

| Use it when | Don't when | Instead use |
|---|---|---|
| exactly one of several options must be chosen | several options can be true at once (a ticket that is both billing *and* technical) | a sigmoid per option |

**Watch out**
- Never write `exp(x) / sum(exp(x))` yourself; large scores overflow. Use `torch.softmax` / `scipy.special.softmax`, which subtract the maximum first.
- 82% is the model's confidence, not accuracy. Models are often confidently wrong; measure calibration on real data.
- The `dim` argument matters. Softmax over the wrong axis of a batch gives numbers that sum to 1 across examples instead of across classes — and no error.